# 自定义层

有时候，我们需要自己发明一个现在深度学习框架中还不存在的层。

## 不带参数的层

首先定义一个没有任何参数的自定义层（减去均值）， 构建他只需要继承基础层类并实现前向传播功能即可

In [1]:
import torch
import torch.nn.functional as F
from torch import nn


class CenteredLayer(nn.Module):
    def __init__(self):
        super().__init__()

    def forward(self, X):
        return X - X.mean()

c:\Users\20249\.conda\envs\test1\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


向该层提供一些数据，验证他是否能按照预期进行工作

In [2]:
layer = CenteredLayer()
layer(torch.FloatTensor([1, 2, 3, 4, 5]))

tensor([-2., -1.,  0.,  1.,  2.])

现在把这个层用在更复杂的模型中

In [3]:
net = nn.Sequential(nn.Linear(8, 128), CenteredLayer())

In [5]:
Y = net(torch.rand(4, 8))
Y.mean()

tensor(2.7940e-09, grad_fn=<MeanBackward0>)

## 带参数的层

这里定义有参数的层，这些参数可以通过训练进行调整。使用内置函数来创建参数。

在这里我们不需要为每个自定义层编写自定义的序列化程序。

现在我们实现自定义版本的全连接层，需要实现两个参数，一个表示权重，一个表示偏置项。

In [ ]:
class MyLinear(nn.Module):
    def __init__(self, in_units, units):
        super().__init__()
        # nn.Parameter 是一个特殊的Tensor包装器，这个张量是网络中的可训练参数，需要在反向传播的时候自动计算梯度并更新
        self.weight = nn.Parameter(torch.randn(in_units, units))
        self.bias = nn.Parameter(torch.randn(units,))
    def forward(self, X):
        linear = torch.matmul(X, self.weight.data) + self.bias.data
        return F.relu(linear)